# Sanad سند — question answering over Saudi law

## Part 1: retrieval baselines

Before training anything: given a question in Arabic, how often do existing retrievers put the right legal text first?

Scored on three question sets: **dev** and **test** (questions an LLM wrote from ArabLegalEval passages) and **najiz** (79 expert-verified questions from real FAQs). The corpus is every legal passage from ArabLegalEval, Najiz and the statute articles cited in ALARB court cases.

Code: https://github.com/readaloush/sanad

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers rank_bm25
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Run

In [ ]:
!python -m sanad.baselines --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$'

## Results

In [ ]:
import json, pandas as pd
r = json.load(open('results/baselines.json'))
print(json.dumps(r['data'], ensure_ascii=False, indent=1))
rows = []
for name, sets in r['results'].items():
    for s, m in sets.items():
        rows.append({'retriever': name, 'set': s, **{k: round(v * 100, 1) for k, v in m.items() if k.startswith('R@')}, 'MRR@10': round(m['MRR@10'], 3)})
pd.DataFrame(rows).pivot(index='retriever', columns='set', values=['R@1', 'R@10', 'MRR@10'])

In [ ]:
!mkdir -p /kaggle/working/out && cp results/baselines.json /kaggle/working/out/
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out